# A/B-тест нового алгоритма рекомендаций

## tl;dr

Эксперимент проводился с 26 июня по 2 июля 2026 года. Контроль — группа 1, новый алгоритм — группа 2.

Глобальный CTR снизился с **0.2096** до **0.2003**, то есть примерно на **−4.46% относительно контроля**. Средний пользовательский CTR почти не изменился, но распределение во второй группе стало резко неоднородным: выросли доли пользователей и с очень низким, и с высоким CTR. Поэтому Welch t-test пользовательского CTR незначим, тогда как Mann–Whitney, пуассоновский bootstrap и оба бакетных теста обнаруживают различия.

**Рекомендация:** не раскатывать алгоритм на всех пользователей. На актуальном потоке основной продуктовый показатель — глобальный CTR — статистически значимо ухудшился. Сначала исследовать сегменты пользователей, для которых алгоритм снижает CTR, и исправить механизм рекомендаций.

## Context & Methods

Основная гипотеза: новый алгоритм в группе 2 увеличивает CTR по сравнению с контрольной группой 1.

Используем:

1. Welch t-test пользовательского CTR;
2. Mann–Whitney U test;
3. Welch t-test сглаженного CTR при `alpha = 5`;
4. Пуассоновский bootstrap глобального CTR;
5. Welch t-test бакетного CTR;
6. Mann–Whitney U test бакетного CTR.

### Key Assumptions

- источник данных зафиксирован явно: `simulator_20260720.feed_actions`;
- пользователь закреплён только за одной экспериментальной группой;
- события за период эксперимента зарегистрированы корректно;
- единица анализа для пользовательских тестов — пользователь;
- порог статистической значимости: `0.05`;
- отсутствие значимости не доказывает полную эквивалентность групп, но не даёт оснований заявлять об улучшении.

## Data

### 1. Импорт библиотек и подключение

In [1]:
import os
from getpass import getpass

import numpy as np
import pandas as pd
import pandahouse
import seaborn as sns
import matplotlib.pyplot as plt

from scipy import stats

sns.set(rc={"figure.figsize": (12, 7)}, style="whitegrid")

DATABASE = "simulator_20260720"

clickhouse_password = os.getenv("CLICKHOUSE_PASSWORD")
if clickhouse_password is None:
    clickhouse_password = getpass("Пароль ClickHouse: ")

connection = {
    "host": "http://clickhouse.lab.karpov.courses:8123",
    "password": clickhouse_password,
    "user": "student",
    "database": DATABASE,
}

### 2. Выгрузка данных на уровне пользователя

In [2]:
query = f'''
SELECT
    exp_group,
    user_id,
    sum(action = 'like') AS likes,
    sum(action = 'view') AS views,
    likes / views AS ctr
FROM {DATABASE}.feed_actions
WHERE toDate(time) BETWEEN '2026-06-26' AND '2026-07-02'
  AND exp_group IN (1, 2)
GROUP BY
    exp_group,
    user_id
'''

df = pandahouse.read_clickhouse(query, connection=connection)
df.head()

,exp_group,user_id,likes,views,ctr
0,1,109963,3,15,0.200000
1,1,26117,32,141,0.226950
2,1,138232,18,73,0.246575
3,1,26295,39,141,0.276596
4,1,18392,7,32,0.218750


### 3. Проверка данных и базовых метрик

In [3]:
data_quality = pd.DataFrame({
    "rows": [len(df)],
    "missing_values": [int(df.isna().sum().sum())],
    "duplicate_user_group_pairs": [
        int(df.duplicated(["exp_group", "user_id"]).sum())
    ],
})

assert set(df["exp_group"]) == {1, 2}
assert (df["views"] > 0).all()
assert df.duplicated(["exp_group", "user_id"]).sum() == 0
assert set(df.loc[df["exp_group"] == 1, "user_id"]).isdisjoint(
    set(df.loc[df["exp_group"] == 2, "user_id"])
)

data_quality

,rows,missing_values,duplicate_user_group_pairs
0,19897,0,0


In [4]:
summary = (
    df.groupby("exp_group")
      .agg(
          users=("user_id", "nunique"),
          likes=("likes", "sum"),
          views=("views", "sum"),
          mean_user_ctr=("ctr", "mean"),
          median_user_ctr=("ctr", "median"),
      )
)

summary["global_ctr"] = summary["likes"] / summary["views"]
summary

,users,likes,views,mean_user_ctr,median_user_ctr,global_ctr
exp_group,,,,,,
1,10020,140339,669543,0.216774,0.205882,0.209604
2,9877,132056,659454,0.216102,0.153285,0.200251


In [5]:
global_ctr_1 = summary.loc[1, "global_ctr"]
global_ctr_2 = summary.loc[2, "global_ctr"]
absolute_effect = global_ctr_2 - global_ctr_1
relative_effect = global_ctr_2 / global_ctr_1 - 1

pd.DataFrame({
    "metric": [
        "Глобальный CTR контроля",
        "Глобальный CTR теста",
        "Абсолютная разница",
        "Относительная разница",
    ],
    "value": [
        global_ctr_1,
        global_ctr_2,
        absolute_effect,
        relative_effect,
    ],
})

,metric,value
0,Глобальный CTR контроля,0.209604
1,Глобальный CTR теста,0.200251
2,Абсолютная разница,-0.009354
3,Относительная разница,-0.044625


## Results

### 4. Сравнение распределений глазами

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.histplot(
    data=df,
    x="ctr",
    hue="exp_group",
    bins=50,
    stat="density",
    common_norm=False,
    palette={1: "royalblue", 2: "tomato"},
    alpha=0.45,
    ax=axes[0],
)
axes[0].set(
    title="Распределение пользовательского CTR",
    xlabel="Пользовательский CTR",
    ylabel="Плотность",
    xlim=(0, 0.7),
)

sns.ecdfplot(
    data=df,
    x="ctr",
    hue="exp_group",
    palette={1: "royalblue", 2: "tomato"},
    ax=axes[1],
)
axes[1].set(
    title="ECDF пользовательского CTR",
    xlabel="Пользовательский CTR",
    ylabel="Доля пользователей",
    xlim=(0, 0.7),
)

plt.tight_layout()
plt.show()

/var/folders/18/23q0sf3n7nqbr62_lkyqgdd80000gn/T/ipykernel_73512/2597910234.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Распределения не совпадают. В тестовой группе заметно больше пользователей как с очень низким, так и с высоким CTR. Из-за разнонаправленных изменений среднее значение почти не меняется, хотя форма распределения существенно отличается.

In [7]:
distribution_diagnostics = (
    df.groupby("exp_group")["ctr"]
      .agg(
          users="count",
          mean="mean",
          std="std",
          median="median",
          share_ctr_below_010=lambda x: (x < 0.10).mean(),
          share_ctr_above_030=lambda x: (x > 0.30).mean(),
      )
)

distribution_diagnostics

,users,mean,std,median,share_ctr_below_010,share_ctr_above_030
exp_group,,,,,,
1,10020,0.216774,0.082969,0.205882,0.042415,0.131836
2,9877,0.216102,0.142870,0.153285,0.266680,0.320036


### 5. Welch t-test и Mann–Whitney U test

In [8]:
ctr_group_1 = df.loc[df["exp_group"] == 1, "ctr"]
ctr_group_2 = df.loc[df["exp_group"] == 2, "ctr"]

raw_ttest = stats.ttest_ind(
    ctr_group_1,
    ctr_group_2,
    equal_var=False,
)

raw_mannwhitney = stats.mannwhitneyu(
    ctr_group_1,
    ctr_group_2,
    alternative="two-sided",
)

pd.DataFrame({
    "test": ["Welch t-test", "Mann–Whitney U"],
    "statistic": [raw_ttest.statistic, raw_mannwhitney.statistic],
    "p_value": [raw_ttest.pvalue, raw_mannwhitney.pvalue],
})

,test,statistic,p_value
0,Welch t-test,4.051492e-01,6.853733e-01
1,Mann–Whitney U,5.518991e+07,4.632206e-45


Welch t-test не обнаруживает различий средних пользовательских CTR: положительные и отрицательные изменения взаимно компенсируются. Mann–Whitney U test, напротив, фиксирует сильный сдвиг распределения (`p-value < 0.001`). Это не противоречие: тесты проверяют разные свойства данных.

### 6. Сглаженный CTR при alpha = 5

In [9]:
alpha = 5

pooled_global_ctr = df["likes"].sum() / df["views"].sum()
df["smoothed_ctr"] = (
    df["likes"] + alpha * pooled_global_ctr
) / (
    df["views"] + alpha
)

smoothed_ctr_group_1 = df.loc[
    df["exp_group"] == 1, "smoothed_ctr"
]
smoothed_ctr_group_2 = df.loc[
    df["exp_group"] == 2, "smoothed_ctr"
]

smoothed_ttest = stats.ttest_ind(
    smoothed_ctr_group_1,
    smoothed_ctr_group_2,
    equal_var=False,
)

pd.DataFrame({
    "statistic": [smoothed_ttest.statistic],
    "p_value": [smoothed_ttest.pvalue],
})

,statistic,p_value
0,1.247575,0.212205


Сглаживание с единым pooled prior уменьшает шум CTR пользователей с небольшим числом просмотров. Welch t-test сглаженного CTR остаётся незначимым, потому что разнонаправленные изменения по-прежнему почти компенсируются в среднем.

### 7. Пуассоновский bootstrap глобального CTR

In [10]:
likes_group_1 = df.loc[df["exp_group"] == 1, "likes"].to_numpy()
views_group_1 = df.loc[df["exp_group"] == 1, "views"].to_numpy()
likes_group_2 = df.loc[df["exp_group"] == 2, "likes"].to_numpy()
views_group_2 = df.loc[df["exp_group"] == 2, "views"].to_numpy()


def poisson_bootstrap(
    likes_1,
    views_1,
    likes_2,
    views_2,
    n_bootstrap=2000,
    batch_size=200,
    seed=42,
):
    rng = np.random.default_rng(seed)
    bootstrap_ctr_1 = []
    bootstrap_ctr_2 = []

    for start in range(0, n_bootstrap, batch_size):
        current_batch_size = min(batch_size, n_bootstrap - start)

        weights_1 = rng.poisson(
            1, size=(current_batch_size, len(likes_1))
        ).astype(np.int8)
        weights_2 = rng.poisson(
            1, size=(current_batch_size, len(likes_2))
        ).astype(np.int8)

        current_ctr_1 = (weights_1 @ likes_1) / (weights_1 @ views_1)
        current_ctr_2 = (weights_2 @ likes_2) / (weights_2 @ views_2)

        bootstrap_ctr_1.append(current_ctr_1)
        bootstrap_ctr_2.append(current_ctr_2)

    return (
        np.concatenate(bootstrap_ctr_1),
        np.concatenate(bootstrap_ctr_2),
    )


bootstrap_ctr_1, bootstrap_ctr_2 = poisson_bootstrap(
    likes_group_1,
    views_group_1,
    likes_group_2,
    views_group_2,
)

bootstrap_difference = bootstrap_ctr_2 - bootstrap_ctr_1
bootstrap_ci = np.quantile(bootstrap_difference, [0.025, 0.975])
opposite_side_count = min(
    (bootstrap_difference <= 0).sum(),
    (bootstrap_difference >= 0).sum(),
)
bootstrap_pvalue = min(
    1,
    2 * (opposite_side_count + 1) / (len(bootstrap_difference) + 1),
)

pd.DataFrame({
    "mean_difference": [bootstrap_difference.mean()],
    "ci_2.5%": [bootstrap_ci[0]],
    "ci_97.5%": [bootstrap_ci[1]],
    "p_value": [bootstrap_pvalue],
})

,mean_difference,ci_2.5%,ci_97.5%,p_value
0,-0.009422,-0.012451,-0.006354,0.001


In [11]:
plt.figure(figsize=(12, 7))

sns.histplot(
    bootstrap_difference,
    bins=40,
    color="slateblue",
)
plt.axvline(
    0,
    color="red",
    linestyle="--",
    label="Отсутствие эффекта",
)
plt.axvline(
    bootstrap_ci[0],
    color="black",
    linestyle=":",
    label="95% доверительный интервал",
)
plt.axvline(
    bootstrap_ci[1],
    color="black",
    linestyle=":",
)
plt.title("Bootstrap-распределение разницы глобального CTR")
plt.xlabel("CTR группы 2 − CTR группы 1")
plt.ylabel("Количество псевдовыборок")
plt.legend()
plt.show()

/var/folders/18/23q0sf3n7nqbr62_lkyqgdd80000gn/T/ipykernel_73512/3404225384.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


95% доверительный интервал разницы полностью ниже нуля. Пуассоновский bootstrap подтверждает статистически значимое снижение глобального CTR; при 2000 итерациях Monte Carlo `p-value < 0.001`.

### 8. Бакетное преобразование

In [12]:
bucket_query = f'''
SELECT
    exp_group,
    bucket,
    sum(likes) / sum(views) AS bucket_ctr,
    quantileExact(0.9)(ctr) AS ctr9
FROM
(
    SELECT
        exp_group,
        xxHash64(user_id) % 50 AS bucket,
        user_id,
        sum(action = 'like') AS likes,
        sum(action = 'view') AS views,
        likes / views AS ctr
    FROM {DATABASE}.feed_actions
    WHERE toDate(time) BETWEEN '2026-06-26' AND '2026-07-02'
      AND exp_group IN (1, 2)
    GROUP BY
        exp_group,
        bucket,
        user_id
)
GROUP BY
    exp_group,
    bucket
'''

bucket_df = pandahouse.read_clickhouse(
    bucket_query,
    connection=connection,
)

bucket_df.groupby("exp_group").agg(
    buckets=("bucket", "nunique"),
    mean_bucket_ctr=("bucket_ctr", "mean"),
    median_bucket_ctr=("bucket_ctr", "median"),
)

,buckets,mean_bucket_ctr,median_bucket_ctr
exp_group,,,
1,50,0.209694,0.209667
2,50,0.200457,0.197906


In [13]:
plt.figure(figsize=(12, 7))

sns.histplot(
    data=bucket_df,
    x="bucket_ctr",
    hue="exp_group",
    bins=15,
    stat="density",
    common_norm=False,
    palette={1: "royalblue", 2: "tomato"},
    alpha=0.5,
)
plt.title("Распределение бакетного CTR")
plt.xlabel("Бакетный CTR")
plt.ylabel("Плотность")
plt.show()

/var/folders/18/23q0sf3n7nqbr62_lkyqgdd80000gn/T/ipykernel_73512/3773687344.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
bucket_ctr_group_1 = bucket_df.loc[
    bucket_df["exp_group"] == 1, "bucket_ctr"
]
bucket_ctr_group_2 = bucket_df.loc[
    bucket_df["exp_group"] == 2, "bucket_ctr"
]

bucket_ttest = stats.ttest_ind(
    bucket_ctr_group_1,
    bucket_ctr_group_2,
    equal_var=False,
)

bucket_mannwhitney = stats.mannwhitneyu(
    bucket_ctr_group_1,
    bucket_ctr_group_2,
    alternative="two-sided",
)

pd.DataFrame({
    "test": [
        "Welch t-test бакетного CTR",
        "Mann–Whitney U бакетного CTR",
    ],
    "statistic": [
        bucket_ttest.statistic,
        bucket_mannwhitney.statistic,
    ],
    "p_value": [
        bucket_ttest.pvalue,
        bucket_mannwhitney.pvalue,
    ],
})

,test,statistic,p_value
0,Welch t-test бакетного CTR,5.614819,4.592645e-07
1,Mann–Whitney U бакетного CTR,1997.000000,2.657643e-07


### 9. Сводная таблица тестов

In [15]:
test_results = pd.DataFrame({
    "method": [
        "Welch t-test пользовательского CTR",
        "Mann–Whitney U пользовательского CTR",
        "Welch t-test сглаженного CTR",
        "Пуассоновский bootstrap",
        "Welch t-test бакетного CTR",
        "Mann–Whitney U бакетного CTR",
    ],
    "p_value": [
        raw_ttest.pvalue,
        raw_mannwhitney.pvalue,
        smoothed_ttest.pvalue,
        bootstrap_pvalue,
        bucket_ttest.pvalue,
        bucket_mannwhitney.pvalue,
    ],
})

test_results["significant_at_0.05"] = (
    test_results["p_value"] < 0.05
)
test_results

,method,p_value,significant_at_0.05
0,Welch t-test пользовательского CTR,6.853733e-01,False
1,Mann–Whitney U пользовательского CTR,4.632206e-45,True
2,Welch t-test сглаженного CTR,2.122055e-01,False
3,Пуассоновский bootstrap,9.995002e-04,True
4,Welch t-test бакетного CTR,4.592645e-07,True
5,Mann–Whitney U бакетного CTR,2.657643e-07,True


## Takeaways

### Почему тесты сработали именно так

Тесты отвечают на разные вопросы и поэтому дают разные результаты. Welch t-test обычного и сглаженного пользовательского CTR сравнивает средние: они почти одинаковы из-за взаимной компенсации положительных и отрицательных эффектов. Mann–Whitney обнаруживает сильное изменение формы распределения. Глобальный CTR, который учитывает число просмотров каждого пользователя, снизился примерно на 4.46%; это ухудшение подтверждают пуассоновский bootstrap и оба бакетных теста.

### Возможная продуктовая ситуация

Новый алгоритм мог разделить аудиторию на два сегмента: части пользователей он стал показывать более релевантные посты, а существенной части — менее релевантные. В тестовой группе доля пользователей с CTR ниже 0.10 выросла примерно с 4% до 27%, одновременно выросла и доля пользователей с CTR выше 0.30. Такое расслоение объясняет незначимый t-test среднего пользовательского CTR и одновременно значимые непараметрические, bootstrap- и бакетные тесты.

### Рекомендация

Не раскатывать алгоритм на всех пользователей. Основная гипотеза об увеличении CTR не подтверждается; напротив, глобальный CTR статистически значимо снизился.

Перед следующим экспериментом:

1. выделить сегменты, в которых CTR просел сильнее всего;
2. проверить связь эффекта с числом просмотров и активностью пользователя;
3. проверить guardrail-метрики: просмотры, активность и retention;
4. проверить фактический контакт пользователей с новым алгоритмом;
5. после исправления алгоритма повторить эксперимент с заранее заданными MDE и primary metric.